# **Computing $\mathcal{R}_0$ with `pyR0compute`**

This notebook presents several examples of the symbolic computation of the ***basic reproduction number*** using the **next-generation matrix** method (van den Driessche and Watmough, 2002), in which $\mathcal{R}_0 = \rho(FV^{-1})$.

The user only needs to write the **ODE system** and specify the **infected compartments**. The library automatically recognizes as parameters all symbols that are neither state variables nor auxiliary expressions, computes the disease-free equilibrium, and separates the new-infection terms ($\mathcal{F}$) from the transition terms ($\mathcal{V}$). Since this decomposition is not unique, the automatic classification can be overridden with `new_infections={...}`, and the equilibrium, when it is not determined by the equations, can be set with `dfe={...}`.

The examples cover classical epidemiological models, vector-borne transmission, within-host dynamics, and strain competition. In addition, normalized sensitivity indices are computed and reports of the procedure are generated as $\LaTeX$ code, ready to paste into a document.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Yvan-BM-Git/pyR0compute/blob/main/examples/pyR0compute_examples.ipynb)

In [ ]:
%pip install pyR0compute                      # Installs the pyR0compute library from PyPI

In [ ]:
import sympy as sp                            # Imports SymPy for symbolic computation
from pyr0compute import R0Model               # Imports the R0Model class from the pyr0compute package
from IPython.display import Markdown, display # Imports Markdown and display to render results as Markdown in Jupyter Notebook

## 1. SEIR model with vital dynamics

$$
\begin{cases}
\dfrac{dS}{dt} = \Lambda - \beta SI - \mu S \\[6pt]
\dfrac{dE}{dt} = \beta SI - (\sigma + \mu)E \\[6pt]
\dfrac{dI}{dt} = \sigma E - (\gamma + \mu)I \\[6pt]
\dfrac{dR}{dt} = \gamma I - \mu R
\end{cases}
$$

Only the equations and the infected compartments are provided.

In [ ]:
seir = R0Model("""
    dS/dt = Lambda - beta*S*I - mu*S
    dE/dt = beta*S*I - (sigma + mu)*E
    dI/dt = sigma*E - (gamma + mu)*I
    dR/dt = gamma*I - mu*R
""", infected=["E", "I"])

seir.R0

In [ ]:
print("Detected parameters:", seir.parameters)
print("Disease-free equilibrium:", seir.dfe)

The matrices $F$, $V$ and $K = FV^{-1}$ are available as SymPy objects:

In [ ]:
seir.F, seir.V, seir.K

In [ ]:
display(Markdown(seir.report_latex("markdown")))

In [ ]:
print(seir.report_latex())

In [ ]:
print(seir.latex())

### Sensitivity indices

$\Upsilon_p = \dfrac{\partial R_0}{\partial p}\dfrac{p}{R_0}$: a 1% increase in $p$ changes $R_0$ by $\Upsilon_p$%.

In [ ]:
seir.sensitivity_indices()

In [ ]:
values = {"Lambda": 10, "beta": 0.001, "sigma": 0.2, "gamma": 0.1, "mu": 0.02}
print("R0 =", seir.R0_numeric(values))
seir.sensitivity_indices(values)

## **2. Classical SIR model (closed population)**

$$
\begin{cases}
\dfrac{dS}{dt}
= -\dfrac{\beta S I}{N} \\[8pt]

\dfrac{dI}{dt}
= \dfrac{\beta S I}{N} - \gamma I \\[8pt]

\dfrac{dR}{dt}
= \gamma I
\end{cases}
$$

### Model conditions

- **Infected compartment:** $I$.
- **Disease-free equilibrium (DFE):**
  $$S = N,\qquad I = 0,\qquad R = 0.$$

In [ ]:
sir = R0Model({"S": "-beta*S*I/N",
               "I": "beta*S*I/N - gamma*I",
               "R": "gamma*I"}, infected=["I"], dfe={"S": "N"})
sir.R0

Without `dfe`, the error message states exactly what is missing:

In [ ]:
try:
    R0Model({"S": "-beta*S*I/N", 
             "I": "beta*S*I/N - gamma*I", 
             "R": "gamma*I"}, infected=["I"])
except Exception as e:
    print(type(e).__name__, "->", e)

## **3. Vector-borne disease: Ross-Macdonald model**

$$
\begin{cases}
\dfrac{dS_h}{dt}
= L_h - \dfrac{a b_h S_h I_v}{N_h} - \mu_h S_h \\[8pt]

\dfrac{dI_h}{dt}
= \dfrac{a b_h S_h I_v}{N_h}
- (\gamma + \mu_h) I_h \\[8pt]

\dfrac{dR_h}{dt}
= \gamma I_h - \mu_h R_h \\[8pt]

\dfrac{dS_v}{dt}
= L_v - \dfrac{a b_v S_v I_h}{N_h} - \mu_v S_v \\[8pt]

\dfrac{dI_v}{dt}
= \dfrac{a b_v S_v I_h}{N_h} - \mu_v I_v
\end{cases}
$$

### State variables

- $S_h$: number of **susceptible hosts**.
- $I_h$: number of **infected hosts**.
- $R_h$: number of **recovered hosts**.
- $S_v$: number of **susceptible vectors**.
- $I_v$: number of **infected vectors**.
- $N_h = S_h + I_h + R_h$: total host population.


In [ ]:
rm = R0Model("""
    dS_h/dt = L_h - a*b_h*S_h*I_v/N_h - mu_h*S_h
    dI_h/dt = a*b_h*S_h*I_v/N_h - (gamma + mu_h)*I_h
    dR_h/dt = gamma*I_h - mu_h*R_h
    dS_v/dt = L_v - a*b_v*S_v*I_h/N_h - mu_v*S_v
    dI_v/dt = a*b_v*S_v*I_h/N_h - mu_v*I_v
""", infected=["I_h", "I_v"])
rm.new_infections

In [ ]:
rm.K

In [ ]:
rm.R0

## **4. Host-vector model with human-to-human transmission and logistic vector growth**

$$
\begin{cases}
N_h = S_h + E_h + I_h + R_h \\[6pt]

\dfrac{dS_h}{dt}
= \Delta_h
- \dfrac{\delta \alpha_{hv} S_h I_v}{N_h}
- \dfrac{\beta \alpha_{hh} S_h I_h}{N_h}
- (\theta + \mu_h)S_h \\[8pt]

\dfrac{dE_h}{dt}
= \dfrac{\delta \alpha_{hv} S_h I_v}{N_h}
+ \dfrac{\beta \alpha_{hh} S_h I_h}{N_h}
- (\sigma + \mu_h)E_h \\[8pt]

\dfrac{dI_h}{dt}
= \sigma E_h
- (\gamma + \mu_h)I_h \\[8pt]

\dfrac{dR_h}{dt}
= \gamma I_h
- \mu_h R_h
+ \theta S_h \\[8pt]

\dfrac{dM_v}{dt}
= \phi_v M_v
\left(1-\dfrac{M_v}{\kappa_v}\right)
- (\omega+\mu_m)M_v \\[8pt]

\dfrac{dS_v}{dt}
= \omega M_v
- \dfrac{\delta\alpha_{vh}S_vI_h}{N_h}
- (\mu_v+\zeta)S_v \\[8pt]

\dfrac{dE_v}{dt}
= \dfrac{\delta\alpha_{vh}S_vI_h}{N_h}
- (\psi+\mu_v+\zeta)E_v \\[8pt]

\dfrac{dI_v}{dt}
= \psi E_v
- (\mu_v+\zeta)I_v
\end{cases}
$$

### State variables

#### Human hosts

- $S_h$: number of **susceptible human hosts**.
- $E_h$: number of **exposed human hosts**, infected but not yet infectious.
- $I_h$: number of **infected and infectious human hosts**.
- $R_h$: number of **recovered human hosts**.
- $N_h = S_h + E_h + I_h + R_h$: **total human population**.

#### Vectors

- $M_v$: population of **immature vectors**.
- $S_v$: number of **susceptible adult vectors**.
- $E_v$: number of **exposed adult vectors**, infected but not yet infectious.
- $I_v$: number of **infected and infectious adult vectors**.

The total human population is defined as an auxiliary expression. The model has two disease-free equilibria (with and without vectors); the one with vectors is selected, and both remain available in `hv.dfe_candidates`.

In [ ]:
hv = R0Model("""
    N_h = S_h + E_h + I_h + R_h
    dS_h/dt = Delta_h - delta*alpha_hv*S_h*I_v/N_h - beta*alpha_hh*S_h*I_h/N_h - (theta + mu_h)*S_h
    dE_h/dt = delta*alpha_hv*S_h*I_v/N_h + beta*alpha_hh*S_h*I_h/N_h - (sigma + mu_h)*E_h
    dI_h/dt = sigma*E_h - (gamma + mu_h)*I_h
    dR_h/dt = gamma*I_h - mu_h*R_h + theta*S_h
    dM_v/dt = phi_v*M_v*(1 - M_v/kappa_v) - (omega + mu_m)*M_v
    dS_v/dt = omega*M_v - delta*alpha_vh*S_v*I_h/N_h - (mu_v + zeta)*S_v
    dE_v/dt = delta*alpha_vh*S_v*I_h/N_h - (psi + mu_v + zeta)*E_v
    dI_v/dt = psi*E_v - (mu_v + zeta)*I_v
""", infected=["E_h", "I_h", "E_v", "I_v"])
hv.dfe

Only $E_h$ and $E_v$ receive new infections, so $R_0$ is the spectral radius of the next-generation matrix with small domain, of size $2\times 2$:

In [ ]:
hv.next_generation_matrix_small

In [ ]:
hv.R0

## **5. Within-host model: target cells, infected cells and virus**

The model describes the dynamics of a viral infection within the host, considering three populations: target cells susceptible to infection, infected cells, and free viral particles.

$$
\begin{cases}
\dfrac{dT}{dt}
= s-dT-\beta TV \\[8pt]

\dfrac{dI}{dt}
= \beta TV-\delta I \\[8pt]

\dfrac{dV}{dt}
= pI-cV
\end{cases}
$$

### State variables

* $T$: number or concentration of **target cells susceptible** to infection.
* $I$: number or concentration of **infected cells**.
* $V$: number or concentration of **free viral particles**.


In [ ]:
tiv = R0Model("""
    dT/dt = s - d*T - beta*T*V
    dI/dt = beta*T*V - delta*I
    dV/dt = p*I - c*V
""", infected=["I", "V"])
tiv.R0

## **6. van den Driessche and Watmough (2002), model with treatment (§4.1)**

$$
\begin{cases}
N = S + E + I + T \\[6pt]

\dfrac{dE}{dt}
= \dfrac{\beta_1 S I}{N}
+ \dfrac{\beta_2 T I}{N}
- (d+\nu+r_1)E
+ p r_2 I \\[8pt]

\dfrac{dI}{dt}
= \nu E
- (d+r_2)I \\[8pt]

\dfrac{dS}{dt}
= b-dS
- \dfrac{\beta_1 S I}{N} \\[8pt]

\dfrac{dT}{dt}
= -dT
+ r_1E
+ q r_2I
- \dfrac{\beta_2 T I}{N}
\end{cases}
$$

### State variables

- $S$: number of **susceptible individuals**.
- $E$: number of **exposed individuals**, infected but not yet infectious.
- $I$: number of **infected and infectious individuals**.
- $T$: number of **individuals under treatment**.
- $N = S+E+I+T$: **total population**.

In [ ]:
tb = R0Model("""
    N = S + E + I + T
    dE/dt = beta1*S*I/N + beta2*T*I/N - (d + nu + r1)*E + p*r2*I
    dI/dt = nu*E - (d + r2)*I
    dS/dt = b - d*S - beta1*S*I/N
    dT/dt = -d*T + r1*E + q*r2*I - beta2*T*I/N
""", infected=["E", "I"])
tb.R0

## **7. Two strains with superinfection**

The model considers a susceptible population and two infectious strains that compete with each other. The interaction between strains is represented by a **superinfection** term $\nu I_1 I_2$. The dominant strain depends on the epidemiological parameters, so the basic reproduction number of the system is given by the larger of the two strain-specific reproduction numbers.

$$
\begin{cases}
\dfrac{dI_1}{dt}
= \beta_1 I_1 S
- (b+\gamma_1)I_1
+ \nu I_1 I_2 \\[8pt]

\dfrac{dI_2}{dt}
= \beta_2 I_2 S
- (b+\gamma_2)I_2
- \nu I_1 I_2 \\[8pt]

\dfrac{dS}{dt}
= b-bS
+\gamma_1 I_1
+\gamma_2 I_2
-(\beta_1I_1+\beta_2I_2)S
\end{cases}
$$

### State variables

- $S$: proportion or number of **susceptible individuals**.
- $I_1$: proportion or number of individuals **infected with strain 1**.
- $I_2$: proportion or number of individuals **infected with strain 2**.

The strain with the larger reproduction number has an epidemiological advantage and, under the model assumptions, determines which of the two strains can become established in the population.

In [ ]:
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    two = R0Model("""
        dI1/dt = beta1*I1*S - (b + gamma1)*I1 + nu*I1*I2
        dI2/dt = beta2*I2*S - (b + gamma2)*I2 - nu*I1*I2
        dS/dt = b - b*S + gamma1*I1 + gamma2*I2 - (beta1*I1 + beta2*I2)*S
    """, infected=["I1", "I2"])
    R0 = two.R0
R0

## **8. CHALLENGE: Compute $R_0$ for your own model**

Now apply the method to an **epidemiological model of your choice**. Replace the equations in the example with those of your model and specify the compartments you consider infected.

If `R0Model` does not correctly identify the **new-infection** and **transition** terms, you can define them explicitly with `new_infections={...}`. Likewise, if you need to set the **disease-free equilibrium (DFE)** manually, you can use `dfe={...}`.

In [ ]:
model = R0Model("""
    dS/dt = ...
    dI/dt = ...
    dR/dt = ...
""", infected=["I"])
model.R0

In [ ]:
print(model.latex())